# RepoClosure: Graph-Guided, Explainable Code Retrieval

**Gemma 4 Developer Agent Paper Track — Best New Resource submission**

RepoClosure is an open-source retrieval toolkit that improves how AI coding agents find relevant code in complex software repositories. It combines structural dependency graphs, semantic embeddings, and lexical retrieval with explainable graph-guided expansion.

This notebook demonstrates:
1. Installing RepoClosure
2. Indexing a Python repository into a typed provenance graph
3. Searching with lexical, semantic, hybrid, and graph-guided methods
4. Inspecting explainable graph-based results

## 1. Install RepoClosure

Install from the public GitHub repository.

In [ ]:
!pip install -q git+https://github.com/thatjelvin/RepoClosure.git

## 2. Index a Repository

Parse a Python repository into a typed, directed provenance graph (module → class → function → method) with AST-derived relationships.

In [ ]:
from repoclosure.indexing import build_index

# Index a small public repository
example_repo = "https://github.com/pallets/flask"
index = build_index(example_repo)

print(f"Indexed {len(index.entities)} entities across {len(index.files)} files")
print(f"Graph nodes: {index.graph.number_of_nodes()}, edges: {index.graph.number_of_edges()}")

## 3. Search for Code

Query the index with natural language. Compare retrieval methods: lexical (BM25), semantic (embeddings), hybrid (RRF fusion), and graph-guided expansion.

In [ ]:
from repoclosure import search

query = "how to handle HTTP exceptions and error responses"

for method in ["lexical", "semantic", "hybrid", "graph_hybrid"]:
    result = index.search(query, method=method, top_k=5, granularity="symbol")
    print(f"\n=== {method.upper()} ===")
    for i, item in enumerate(result.entities[:5], 1):
        print(f"  {i}. {item.symbol}  ({item.file}:{item.line})  score={item.score:.3f}")

## 4. Explainable Graph-Guided Results

The `graph_hybrid` method returns graph explanations showing why each entity was selected — distinguishing statically verified AST edges from heuristics.

In [ ]:
result = index.search(query, method="graph_hybrid", top_k=3, granularity="symbol")

for i, item in enumerate(result.entities, 1):
    print(f"\n{i}. {item.symbol}")
    print(f"   File: {item.file}:{item.line}")
    print(f"   Fused score: {item.score:.3f}")
    if hasattr(item, "graph_explanation") and item.graph_explanation:
        print(f"   Graph explanation:")
        for edge in item.graph_explanation:
            print(f"     - {edge['source']} → {edge['target']} ({edge['edge_type']}, verified={edge.get('verified', 'N/A')})")

## 5. CLI Usage

RepoClosure also includes a command-line interface for indexing and searching repositories.

In [ ]:
!repoclosure --help

## Resources

- **Repository:** https://github.com/thatjelvin/RepoClosure
- **License:** Apache 2.0
- **Competition:** Gemma 4 Developer Agent Paper Track (Best New Resource)

This notebook is part of the RepoClosure submission. All code is open source and reproducible.